## IR Spectra Exercise (Notebook)

Complete the TODOs in this notebook (and/or in `exercise/src/student.py`).

### What to produce
- At least one overlay plot: **one query** + **top-3 references** (raw)
- At least one before/after plot: **raw vs smoothed**
- A top-k ranking table for each query using at least **two** similarity methods

### How you will be evaluated
- `pytest exercise/tests`
- Quality of plots + reasoning + implementation efficiency


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from exercise.src.student import (
    load_spectrum_json,
    ensure_increasing_x,
    resample_to_grid,
    smooth_savgol,
    smooth_gaussian,
    similarity_cosine,
    similarity_pearson,
    rank_queries,
)

DATA_DIR = Path("exercise/data")
REF_DIR = DATA_DIR / "reference"
QUERY_DIR = DATA_DIR / "queries"
EXPECTED_DIR = Path("exercise/expected")


In [ ]:
# Load data
references = [load_spectrum_json(p) for p in sorted(REF_DIR.glob("*.json"))]
queries = [load_spectrum_json(p) for p in sorted(QUERY_DIR.glob("*.json"))]

print(f"Loaded {len(references)} reference spectra")
print(f"Loaded {len(queries)} query spectra")
print("Example ref:", references[0].spectrum_id, references[0].label, references[0].x_cm1.shape)


### 1) Visualize raw spectra

Pick **one query** spectrum and plot it against a few references to get intuition about noise, baseline drift, and shifts.


In [ ]:
q = queries[0]

plt.figure(figsize=(10, 4))
plt.plot(q.x_cm1, q.y, label=f"query {q.spectrum_id} ({q.label})", lw=2)
for r in references[:3]:
    plt.plot(r.x_cm1, r.y, label=f"ref {r.label}", alpha=0.8)
plt.gca().invert_xaxis()  # conventional IR plots often show high->low cm^-1
plt.xlabel("Wavenumber (cm$^{-1}$)")
plt.ylabel("Intensity")
plt.title("Raw spectra (example)")
plt.legend()
plt.tight_layout()
plt.show()


### 2) Smoothing

Implement at least two smoothers in `exercise/src/student.py`:
- Savitzky–Golay (`smooth_savgol`)
- Gaussian (`smooth_gaussian`)

Then compare raw vs smoothed.


In [ ]:
y_sg = smooth_savgol(q.y, window_length=31, polyorder=3)
y_gauss = smooth_gaussian(q.y, sigma=2.0)

plt.figure(figsize=(10, 4))
plt.plot(q.x_cm1, q.y, label="raw", alpha=0.6)
plt.plot(q.x_cm1, y_sg, label="savgol", lw=2)
plt.plot(q.x_cm1, y_gauss, label="gaussian", lw=2)
plt.gca().invert_xaxis()
plt.xlabel("Wavenumber (cm$^{-1}$)")
plt.ylabel("Intensity")
plt.title(f"Smoothing comparison ({q.spectrum_id})")
plt.legend()
plt.tight_layout()
plt.show()


### 3) Similarity + ranking

Implement in `exercise/src/student.py`:
- `similarity_cosine`
- `similarity_pearson`
- `rank_queries`

Then compute top-3 matches for all queries.


In [ ]:
rankings_cos = rank_queries(queries, references, method="cosine", top_k=3)
rankings_pear = rank_queries(queries, references, method="pearson", top_k=3)

rows = []
for q in queries:
    top_cos = rankings_cos[q.spectrum_id]
    top_pear = rankings_pear[q.spectrum_id]
    rows.append(
        {
            "query_id": q.spectrum_id,
            "true_label": q.label,
            "cos_top1": top_cos[0][0],
            "cos_top1_score": top_cos[0][1],
            "pear_top1": top_pear[0][0],
            "pear_top1_score": top_pear[0][1],
        }
    )

df = pd.DataFrame(rows)
df

### 4) Overlay plot (query + top-3)

Pick one query, compute top-3 (cosine), and plot the query against those 3 references.


In [ ]:
q = queries[1]
top3 = rankings_cos[q.spectrum_id]
labels = [lab for (lab, _) in top3]

ref_by_label = {r.label: r for r in references}

plt.figure(figsize=(10, 4))
plt.plot(q.x_cm1, q.y, label=f"query {q.spectrum_id} ({q.label})", lw=2)
for lab in labels:
    r = ref_by_label[lab]
    plt.plot(r.x_cm1, r.y, label=f"ref {lab}", alpha=0.9)

plt.gca().invert_xaxis()
plt.xlabel("Wavenumber (cm$^{-1}$)")
plt.ylabel("Intensity")
plt.title(f"Overlay: {q.spectrum_id} + top-3 (cosine)")
plt.legend()
plt.tight_layout()
plt.show()
